# Cảnh báo sớm hạ SpO₂ với ngưỡng do bác sĩ đặt: model hybrid nhận ngưỡng T

## Tóm tắt
Một model dùng cho mọi ngưỡng:

$$f(\text{60 phút lịch sử},\ T,\ \text{baseline}) \rightarrow \text{logit của } P(\text{SpO₂} < T \text{ liên tục} \ge 3 \text{ phút trong 30 phút tới})$$

Bác sĩ đổi T (COPD 88–92%, đa số bệnh khác 94–98%, theo BTS 2017) thì chỉ gửi config mới xuống vòng đeo tay, **không train lại**. Mọi đầu vào được biểu diễn **tương đối so với T** để model học quy luật "cách ngưỡng bao xa, đang đi xuống nhanh thế nào" thay vì một con số tuyệt đối. Model chạy trên ESP32-C3 dạng TFLite Micro INT8.

## Dữ liệu
- MIMIC-III Waveform Matched Subset, numerics, **chỉ record 1 Hz** (D1), lưới 1 phút. Nhãn được làm sạch bằng HR từ ECG (D2).
- `dataset_tc/{train,val,test}.npz`: cửa sổ 60 phút × 4 kênh, nhãn `Y[i, T]` và cờ `OK[i, T]` (đầu vào chưa có sự kiện < T) cho T ∈ {86,…,94}, `baseline` (trung vị SpO₂ 6–24 giờ trước cửa sổ) và `baseline_hours`. Hợp đồng dữ liệu ghi ở đầu `tc_core.py`.
- **T = 89 và 91 không dùng để train**, chỉ để kiểm tra khả năng nội suy sang ngưỡng chưa thấy.
- Chia theo bệnh nhân 70/15/15.

## Phương pháp
| Bước | Kỹ thuật | Lý do |
|---|---|---|
| Đầu vào chuỗi | `spo2_mean − T`, `spo2_min − T`, `spo2_valid`, `pulse_mean` + 2 kênh mask; z-score theo train (sau khi đã trừ T) | học quy luật tương đối, không học con số 88 |
| Đầu vào bảng | 23 đặc trưng: tương đối với T, xu hướng, mạch, baseline, chất lượng, T | tín hiệu chính là "khoảng cách tới ngưỡng" |
| Lấy mẫu T | mỗi cửa sổ một T ngẫu nhiên mỗi epoch (không nhân bản ×7), chỉ tính loss khi `OK[i, T]` | dữ liệu không phình, mọi T được học đều |
| Mất cân bằng | giữ toàn bộ mẫu dương, lấy 30% mẫu âm với trọng số bù 1/0,3 | nhanh hơn mà loss vẫn không lệch |
| Loss | BCE (logit) + λ·relu(logit_T − logit_{T+1}) | ép tính đơn điệu: T cao hơn thì rủi ro không thể thấp hơn |
| Dừng sớm | trung bình AUPRC trên các T train của val; 3 seed; checkpoint mỗi epoch | |
| Ngưỡng cảnh báo | chọn trên val theo **báo giả/ngày thật** (sim_alert, sau persistence + cooldown), bảng T → ngưỡng, làm mượt theo T | test giữ nguyên ngưỡng |
| Triển khai | TFLite INT8, 2 đầu vào, đầu ra logit; `meta.json` mang bảng ngưỡng | firmware chỉ cần bảng, không cần train |

## Quy ước đánh giá
- Val để dừng sớm, chọn seed, chọn ngưỡng, chọn persistence. **Test chỉ mở một lần** (mục 9).
- Khoảng tin cậy 95% bootstrap **theo bệnh nhân**; so sánh model bằng chênh AUPRC theo cặp.
- Báo giả báo cáo là **cảnh báo thật** sau persistence và cooldown; con số đếm theo cửa sổ chỉ là cận trên.

## Danh mục đồ thị
G1 AUPRC theo T · G2 PR/ROC theo T · G3 bắt sự kiện vs báo giả/ngày so với luật · G4 đơn điệu · G5 hiệu chuẩn · G6 sim_alert theo T · G7 báo trước và báo giả theo bệnh nhân · G8 ablation · G9 INT8 vs float · G10 tầng Critical · (G11 ở `10_lightgbm_tc.ipynb`)

## Mục lục
1 Cấu hình · 2 Dữ liệu · 3 Mô hình · 4 Huấn luyện · 5 Đánh giá trên val · 6 Ablation · 7 Mô phỏng cảnh báo trên val · 8 Chốt cấu hình · 9 Test (một lần) · 10 Xuất model · 11 Suy luận thử · 12 Tổng kết

## 1. Cấu hình
| Nhóm | Tham số | Ý nghĩa |
|---|---|---|
| Đường dẫn | `DATA_DIR`, `CLEAN_DIR`, `RUN_ROOT`, `EXPORT_DIR` | dữ liệu cửa sổ; lưới 1 phút (cho mô phỏng); checkpoint; model xuất |
| Ngưỡng | `T_TRAIN`, `T_HOLDOUT`, `T_RANGE` | T dùng để train; T để riêng kiểm tra nội suy; khoảng T hợp lệ trên web/firmware |
| Huấn luyện | `SEEDS`, `BATCH`, `MAX_EPOCHS`, `PATIENCE`, `LR`, `NEG_RATE`, `LAMBDA_MONO` | `NEG_RATE`: tỷ lệ giữ mẫu âm mỗi epoch; `LAMBDA_MONO`: hệ số phạt đơn điệu |
| Kiến trúc | `FILTERS`, `KERNEL`, `DROPOUT`, `TAB_UNITS` | nhánh chuỗi và nhánh đặc trưng |
| So sánh | `BASELINES` | điểm LightGBM đã lưu bởi `10_lightgbm_tc.ipynb`; luật được tính ngay trong notebook |
| Cảnh báo | `TIERS` | ngân sách **báo giả/ngày thật**, persistence `k/n`, cooldown (phút) của tầng Watch và Alert |
| Critical | `CRIT`, `CRIT_DEPLOY` | các ngưỡng Critical để thống kê (G10) và cấu hình mặc định gửi xuống thiết bị |

**Phục hồi:** chạy lại notebook từ đầu, mỗi run tự tiếp tục từ epoch cuối đã lưu. Không đổi `CFG` giữa chừng; thử cấu hình mới thì đổi `RUN_ROOT`.

In [ ]:
import os, json, time, numpy as np, pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tqdm.auto import tqdm
from IPython.display import display
from sklearn.metrics import average_precision_score, roc_auc_score
import tc_core as C, eval_utils as E, sim_alert as SA

CFG = dict(
    DATA_DIR="../data/data/dataset_tc", CLEAN_DIR="../data/data/clean_1min",
    RUN_ROOT="runs/hybrid_tc_v1", EXPORT_DIR="models_v3", NAME="hybrid_tc",
    T_TRAIN=C.T_TRAIN, T_HOLDOUT=C.T_HOLDOUT, T_RANGE=C.T_RANGE,
    SEEDS=(1, 2, 3), BATCH=512, MAX_EPOCHS=40, PATIENCE=6, LR=1e-3,
    NEG_RATE=0.3, LAMBDA_MONO=1.0,
    FILTERS=(16, 32, 32), KERNEL=5, DROPOUT=0.2, TAB_UNITS=32,
    BASELINES=dict(lgbm="runs/lgbm_tc"),
    TIERS=dict(watch=dict(fa_day=3.0, k=1, n=1, cooldown=30), alert=dict(fa_day=1.0, k=1, n=1, cooldown=60)),
    FA_GRID=(0.25, 0.5, 1, 2, 4),                 # ngân sách báo giả/ngày để vẽ G3
    THR_SMOOTH="isotonic",                        # làm mượt bảng T -> ngưỡng: isotonic | linear | none
    CRIT=dict(crit_T=(80, 82, 85, 88), n_min=3, sqi_min=0.5, hr_low=40, hr_high=140, cooldown=30),
    CRIT_DEPLOY=dict(crit_T=85, n_min=3, sqi_min=0.5, hr_low=40, hr_high=140, cooldown=30),
    RUN_ABLATION=True, ABL_SEED=1,
)
ROOT = CFG["RUN_ROOT"]; os.makedirs(ROOT, exist_ok=True)
T_TR, T_HO = [int(t) for t in CFG["T_TRAIN"]], [int(t) for t in CFG["T_HOLDOUT"]]
sig = lambda z: 1 / (1 + np.exp(-z))
print("TensorFlow", tf.__version__, "| CPU:", os.cpu_count(), "| GPU:", tf.config.list_physical_devices("GPU"))

## 2. Dữ liệu

### 2.1 Nạp cửa sổ và thống kê nhãn theo T
Mỗi tập có `X` (n, 60, 4), `Y` và `OK` (n, K) theo `T_list`, `baseline`, `baseline_hours`, `pid`, `rec`, `start`. Bảng dưới là **"bảng số cửa sổ và tỷ lệ dương theo T"**: tỷ lệ dương tăng mạnh theo T, nên AUPRC tuyệt đối ở các T khác nhau không so sánh trực tiếp được; hãy đọc kèm hệ số so với ngẫu nhiên.

Phần không phụ thuộc T của đặc trưng (mức SpO₂, xu hướng, mạch, chất lượng, bảng `frac_below` cho mọi ngưỡng nguyên) được tính một lần và lưu ở `features_tc/`; đặc trưng tại một T cụ thể được ghép lại rất nhanh trong lúc train.

In [ ]:
DS = {k: C.load_split(CFG["DATA_DIR"], k) for k in ("train", "val", "test")}
T_LIST = DS["val"]["T_list"]
assert all((DS[k]["T_list"] == T_LIST).all() for k in DS) and set(T_TR + T_HO) <= set(T_LIST.tolist())
for a, b in (("train", "val"), ("train", "test"), ("val", "test")):
    assert not set(DS[a]["pid"]) & set(DS[b]["pid"]), f"Có bệnh nhân nằm ở cả {a} và {b}"
FEAT_DIR = os.path.join(os.path.dirname(CFG["DATA_DIR"].rstrip("/")), "features_tc")
ST = {k: C.stats_cache(DS[k], f"{FEAT_DIR}/{k}_stats.npz") for k in DS}

LT = C.label_table(DS, T_LIST)
display(LT.pivot(index="T", columns="split", values=["n_ok", "n_pos", "prevalence", "pat_pos"]).round(4))
fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
for k, g in LT.groupby("split", sort=False):
    ax[0].plot(g["T"], g["prevalence"] * 100, "o-", label=k); ax[1].plot(g["T"], g["pat_pos"], "o-", label=k)
for a in ax:
    for t in T_HO: a.axvspan(t - .4, t + .4, color="gray", alpha=.15)
    a.set_xlabel("Ngưỡng T (%)"); a.legend()
ax[0].set(ylabel="% cửa sổ dương", title="Tỷ lệ dương theo T (nền xám: T không train)"); ax[1].set(ylabel="bệnh nhân", title="Số bệnh nhân có cửa sổ dương")
plt.tight_layout(); plt.show()
bt, bh = DS["train"]["baseline"], DS["train"]["baseline_hours"]
print(f"Cửa sổ train có baseline ≥ {C.BASE_MIN_HOURS} giờ: {np.mean(np.isfinite(bt) & (bh >= C.BASE_MIN_HOURS)):.1%} "
      f"| baseline_hours trung vị {np.nanmedian(bh):.1f} giờ (thiếu baseline thì dùng SpO₂ trung bình của chính cửa sổ)")

### 2.2 Unit test đặc trưng tương đối
Kiểm tra trước khi train: (1) đổi T một lượng ΔT thì mọi đặc trưng `*_rT` và `base_rT` dịch đúng −ΔT, các đặc trưng khác giữ nguyên; (2) đường tính nhanh `tab_from_stats` khớp bản tham chiếu `make_features` (bản firmware phải làm theo); (3) chuỗi dịch đúng; (4) loại sự kiện nhiễu bằng HR; (5) `fire_times` vector hóa khớp bản cũ; (6) lấy mẫu T chỉ chọn ô `OK`.

In [ ]:
import test_tc_core as TT
for n_, f_ in vars(TT).items():
    if n_.startswith("test_"): f_()
i_ = np.arange(min(3000, len(DS["val"]["X"]))); dv_ = DS["val"]
F88, F92 = C.tab_from_stats(ST["val"], 88, i_), C.tab_from_stats(ST["val"], 92, i_)
for c in C.TAB_REL[:5] + ["base_rT"]: assert np.allclose((F92[c] - F88[c]).dropna(), -4, atol=1e-4), c
Fref = C.make_features(dv_["X"][i_], 90, dv_["baseline"][i_], dv_["baseline_hours"][i_])
assert np.allclose(C.tab_from_stats(ST["val"], 90, i_).to_numpy(), Fref.to_numpy(), atol=1e-4, equal_nan=True)
print(f"Unit test: OK | {len(C.tab_cols())} đặc trưng bảng:", C.tab_cols())

## 3. Mô hình hybrid nhận T

```text
seq (60×6) → Conv1D(16,k5) → MaxPool → Conv1D(32,k5) → MaxPool → Conv1D(32,k3) → GlobalAvgPool ─┐
tab (23)   → Dense(32, ReLU) ───────────────────────────────────────────────────────────────────┴→ Concat → Dropout → Dense(16) → Dense(1) = logit
```
- Kiến trúc giữ như bản v2; T đi vào qua **cả hai nhánh**: kênh SpO₂ đã trừ T (nhánh chuỗi) và các đặc trưng tương đối + T (nhánh bảng).
- Khoảng 8 nghìn tham số, mục tiêu < 30 KB khi INT8. Đầu ra là logit (sigmoid ở ngoài model).

In [ ]:
def build_hybrid(n_tab, lr=CFG["LR"]):
    L = keras.layers
    s = keras.Input((C.IN_MIN, 6), name="seq"); t = keras.Input((n_tab,), name="tab"); x = s
    for i, f in enumerate(CFG["FILTERS"]):
        x = L.Conv1D(f, CFG["KERNEL"] if i < 2 else 3, padding="same", activation="relu")(x)
        if i < 2: x = L.MaxPooling1D(2)(x)
    z = L.Concatenate()([L.GlobalAveragePooling1D()(x), L.Dense(CFG["TAB_UNITS"], activation="relu")(t)])
    z = L.Dense(16, activation="relu")(L.Dropout(CFG["DROPOUT"])(z))
    m = keras.Model([s, t], L.Dense(1)(z), name="hybrid_tc")                     # đầu ra: logit
    m.compile(keras.optimizers.Adam(lr), keras.losses.BinaryCrossentropy(from_logits=True)); return m
build_hybrid(len(C.tab_cols())).summary()

## 4. Huấn luyện

### 4.1 Thuật toán
Mỗi epoch:
1. `tc_core.sample_epoch`: mỗi cửa sổ nhận **một** T ngẫu nhiên, đều trên các T train mà cửa sổ còn `OK`; giữ mọi mẫu dương, giữ `NEG_RATE` mẫu âm với trọng số `1/NEG_RATE`. Dữ liệu không bị nhân bản 7 lần, mọi T được chọn ngang nhau.
2. Mỗi batch chạy model hai lần, tại T và T+1 trên cùng cửa sổ. Loss = BCE có trọng số tại T + `LAMBDA_MONO` × trung bình `relu(logit_T − logit_{T+1})`.
3. Cuối epoch: chấm val ở **mọi** T. Tiêu chí dừng sớm là trung bình AUPRC trên các T train. Ghi thêm AUPRC trung bình ở T không train (chỉ để theo dõi, không dùng để chọn) và tỷ lệ cặp T kề nhau vi phạm đơn điệu.
4. Lưu `last.keras` (cả trạng thái optimizer), `best.keras`, `state.json`; ghi qua file tạm rồi đổi tên.

Chuẩn hóa (`norm_*.npz`) tính trên train **sau khi** đã đổi sang giá trị tương đối, với T rút ngẫu nhiên trong các T train.

### 4.2 Cách đọc bảng
`loss` là BCE train (có trọng số), `mono` là phạt đơn điệu, `val_loss`/`val_auprc` tính trên val ở các T train, `ho_auprc` ở T không train, `viol` là tỷ lệ cặp T kề nhau có P giảm. `gap = auprc(train, tập con) − val_auprc`. Val chỉ có vài chục bệnh nhân dương ở T thấp nên dao động giữa các epoch là bình thường; đọc theo xu hướng.

In [ ]:
def get_norm(rel=True, use_base=True):
    p = f"{ROOT}/norm_rel{int(rel)}_base{int(use_base)}.npz"
    if os.path.exists(p): return C.load_norm(p)
    nm = C.fit_norm(DS["train"]["X"], ST["train"], T_TR, rel, use_base); C.save_norm(p, nm); return nm

def predict_T(model, k, nm, T, idx=None, bs=8192):
    d = DS[k]; idx = np.arange(len(d["X"])) if idx is None else np.asarray(idx)
    T = np.broadcast_to(np.asarray(T), (len(idx),)); out = np.empty(len(idx), np.float32)
    for a in range(0, len(idx), bs):
        out[a:a + bs] = np.asarray(model.predict_on_batch(C.inputs(d, ST[k], nm, T[a:a + bs], idx[a:a + bs]))).ravel()
    return out

def score_matrix(model, k, nm, T_list=None):
    """Logit (n, K) của mọi cửa sổ ở các T trong T_list (mặc định toàn bộ T_LIST); cột không tính để NaN."""
    S = np.full((len(DS[k]["X"]), len(T_LIST)), np.nan, np.float32)
    for T in (T_LIST if T_list is None else T_list): S[:, C.col(DS[k], T)] = predict_T(model, k, nm, int(T))
    return S

def val_metrics(Z, T_ch):
    d, ls, aps = DS["val"], [], []
    for T in T_ch:
        j = C.col(d, T); ok = d["OK"][:, j]; y = d["Y"][ok, j]; z = Z[ok, j].astype(np.float64)
        ls.append(np.mean(np.maximum(z, 0) - z * y + np.log1p(np.exp(-np.abs(z))))); aps.append(average_precision_score(y, z))
    return float(np.mean(ls)), float(np.mean(aps))

_bce = keras.losses.BinaryCrossentropy(from_logits=True, reduction="none")
def make_step(model, lam):
    opt, var = model.optimizer, model.trainable_variables
    if not opt.built: opt.build(var)
    @tf.function(reduce_retracing=True)
    def step(s0, t0, s1, t1, y, w):
        with tf.GradientTape() as tape:
            z0 = model([s0, t0], training=True)[:, 0]
            l_bce = tf.reduce_sum(w * _bce(y[:, None], z0[:, None])) / tf.reduce_sum(w)
            l_mono = tf.reduce_mean(tf.nn.relu(z0 - model([s1, t1], training=True)[:, 0])) if lam > 0 else tf.constant(0.0)
            loss = l_bce + lam * l_mono
        opt.apply_gradients(zip(tape.gradient(loss, var), var))
        return l_bce, l_mono
    return step

def _save(model, path): tmp = path + ".tmp.keras"; model.save(tmp); os.replace(tmp, path)

def train_run(name, seed, var):
    """var: dict(rel, use_base, fixed_T). fixed_T = số -> model riêng cho một T (ablation), không phạt đơn điệu."""
    run = f"{ROOT}/{name}/s{seed}"; os.makedirs(run, exist_ok=True)
    fixed = var.get("fixed_T"); T_ch = [fixed] if fixed else T_TR; lam = 0.0 if fixed else CFG["LAMBDA_MONO"]
    nm = get_norm(var.get("rel", True), var.get("use_base", True)); d = DS["train"]; B = CFG["BATCH"]
    if os.path.exists(f"{run}/state.json") and os.path.exists(f"{run}/last.keras"):
        st = json.load(open(f"{run}/state.json", encoding="utf-8")); model = keras.models.load_model(f"{run}/last.keras")
        print(f"♻ {name}/s{seed}: tiếp tục từ epoch {st['epoch']} (tốt nhất epoch {st['best_epoch']}, val_auprc {st['best']:.4f})")
    else:
        keras.utils.set_random_seed(seed); model = build_hybrid(len(nm["tab_cols"]))
        st = dict(epoch=0, best=-1.0, best_epoch=-1, wait=0, done=False, history=[])
        json.dump(dict(CFG=CFG, var=var), open(f"{run}/config.json", "w", encoding="utf-8"), indent=1, default=str)
    if st["done"]: print(f"✔ {name}/s{seed}: đã xong (val_auprc {st['best']:.4f} ở epoch {st['best_epoch']})"); return st
    step = make_step(model, lam)
    rs = np.random.default_rng(0); sub = rs.choice(len(d["X"]), min(40000, len(d["X"])), replace=False)
    Tsub = rs.choice(T_ch, len(sub)); jsub = np.array([C.col(d, t) for t in Tsub]); ksub = d["OK"][sub, jsub]
    sub, Tsub, ysub = sub[ksub], Tsub[ksub], d["Y"][sub[ksub], jsub[ksub]]
    outer = tqdm(total=CFG["MAX_EPOCHS"], initial=st["epoch"], desc=f"{name}/s{seed}", unit="epoch")
    outer.write(f"{'Ep':>3} | {'loss':>7} {'mono':>6} {'val_loss':>8} | {'auprc':>6} {'val_auprc':>9} {'ho_auprc':>8} | {'viol':>6} | {'giây':>5} | ghi chú")
    for ep in range(st["epoch"], CFG["MAX_EPOCHS"]):
        t0 = time.time(); rng = np.random.default_rng(seed * 1000 + ep)
        idx, T, y, w = C.sample_epoch(d, T_ch, CFG["NEG_RATE"], rng, fixed_T=fixed)
        lb = lm = 0.0; nb = 0
        for a in tqdm(range(0, len(idx), B), desc=f"Epoch {ep + 1}", leave=False):
            ii, tt = idx[a:a + B], T[a:a + B]
            x0 = C.inputs(d, ST["train"], nm, tt, ii); x1 = C.inputs(d, ST["train"], nm, tt + 1, ii) if lam else x0
            l1, l2 = step(x0["seq"], x0["tab"], x1["seq"], x1["tab"], y[a:a + B], w[a:a + B]); lb += float(l1); lm += float(l2); nb += 1
        Zv = score_matrix(model, "val", nm, T_ch if fixed else None); vl, va = val_metrics(Zv, T_ch)
        r = dict(epoch=ep + 1, loss=lb / nb, mono=lm / nb, val_loss=vl, val_auprc=va, n_samples=int(len(idx)),
                 auprc=float(average_precision_score(ysub, predict_T(model, "train", nm, Tsub, sub))))
        r["gap"] = r["auprc"] - va
        r["ho_auprc"] = E.mean_auprc(Zv, DS["val"], T_HO) if not fixed else np.nan
        r["viol"] = E.monotonic_report(Zv, T_LIST)["adjacent"] if not fixed else np.nan
        r["sec"] = time.time() - t0; H = st["history"]; H.append(r); note = []
        if va > st["best"]:
            st["best"], st["best_epoch"], st["wait"] = va, r["epoch"], 0; _save(model, f"{run}/best.keras"); note.append("★ tốt nhất")
        else: st["wait"] += 1
        if len(H) >= 3 and H[-1]["val_loss"] > H[-2]["val_loss"] > H[-3]["val_loss"] and H[-1]["loss"] < H[-3]["loss"]:
            note.append("⚠ val_loss tăng 2 epoch liền (nghi overfit)")
        st["epoch"] = r["epoch"]; st["done"] = st["wait"] >= CFG["PATIENCE"] or r["epoch"] >= CFG["MAX_EPOCHS"]
        _save(model, f"{run}/last.keras")
        json.dump(st, open(f"{run}/state.json.tmp", "w", encoding="utf-8")); os.replace(f"{run}/state.json.tmp", f"{run}/state.json")
        outer.update(1)
        outer.write(f"{r['epoch']:>3} | {r['loss']:7.4f} {r['mono']:6.4f} {vl:8.4f} | {r['auprc']:6.3f} {va:9.4f} {r['ho_auprc']:8.4f} | "
                    f"{r['viol']:6.4f} | {r['sec']:5.0f} | {' '.join(note)}")
        if st["done"]:
            outer.write("■ Dừng: " + ("hết kiên nhẫn (early stopping)" if st["wait"] >= CFG["PATIENCE"] else "đủ số epoch tối đa")); break
    outer.close(); return st

def load_best(name, s): return keras.models.load_model(f"{ROOT}/{name}/s{s}/best.keras")

def seed_scores(name, s, k, nm, T_list=None):
    """Logit của best.keras trên tập k, có lưu đĩa theo epoch tốt nhất (đổi checkpoint thì tự tính lại)."""
    st = json.load(open(f"{ROOT}/{name}/s{s}/state.json", encoding="utf-8")); p = f"{ROOT}/{name}/s{s}/logit_{k}_e{st['best_epoch']}.npy"
    if os.path.exists(p): return np.load(p)
    Z = score_matrix(load_best(name, s), k, nm, T_list); np.save(p, Z); return Z

### 4.3 Chạy (tự resume)
Model chính: đầu vào tương đối + baseline, một model chung cho mọi T, 3 seed.

In [ ]:
MAIN = dict(rel=True, use_base=True, fixed_T=None)
NM = get_norm(True, True)
RUNS = {s: train_run("main", s, MAIN) for s in CFG["SEEDS"]}

### 4.4 Đường cong học
Bốn biểu đồ, mỗi seed một màu: loss (nét đứt = train, liền = val), AUPRC (đứt = train tập con, liền = val ở T train, chấm = val ở T không train), phạt và tỷ lệ vi phạm đơn điệu, gap AUPRC. Đường chấm dọc là epoch tốt nhất. Đường cong ổn định khi ba seed gần nhau và `viol` giảm về gần 0.

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(22, 4.2))
for i, s in enumerate(CFG["SEEDS"]):
    H = pd.DataFrame(RUNS[s]["history"]); c = f"C{i}"; b = RUNS[s]["best_epoch"]
    ax[0].plot(H.epoch, H.loss, c=c, ls="--"); ax[0].plot(H.epoch, H.val_loss, c=c, label=f"seed {s}")
    ax[1].plot(H.epoch, H.auprc, c=c, ls="--"); ax[1].plot(H.epoch, H.val_auprc, c=c, label=f"seed {s}"); ax[1].plot(H.epoch, H.ho_auprc, c=c, ls=":")
    ax[2].plot(H.epoch, H.mono, c=c, ls="--"); ax[2].plot(H.epoch, H.viol, c=c, label=f"seed {s}")
    ax[3].plot(H.epoch, H.gap, c=c, label=f"seed {s}")
    for a in ax: a.axvline(b, ls=":", c=c, lw=.8)
ax[0].set(title="Loss BCE (đứt = train, liền = val)", xlabel="epoch")
ax[1].set(title="AUPRC (đứt = train, liền = val T train, chấm = val T không train)", xlabel="epoch")
ax[2].set(title="Đơn điệu (đứt = phạt train, liền = tỷ lệ vi phạm val)", xlabel="epoch", yscale="symlog", ylim=(0, None))
ax[3].set(title="Gap AUPRC = train − val", xlabel="epoch"); ax[3].axhline(0, c="k", lw=.5)
for a in ax: a.legend(fontsize=8)
plt.tight_layout(); plt.show()
display(pd.DataFrame({f"seed {s}": dict(epoch_tot_nhat=RUNS[s]["best_epoch"], val_auprc_T_train=round(RUNS[s]["best"], 4),
                                        so_epoch=RUNS[s]["epoch"]) for s in CFG["SEEDS"]}).T)

## 5. Đánh giá trên val

### 5.1 AUPRC theo T (G1) và PR/ROC (G2)
- Mỗi seed dùng `best.keras`; **ensemble** = trung bình xác suất 3 seed.
- Đối chứng: `lgbm` (LightGBM có ràng buộc đơn điệu, điểm lưu bởi `10_lightgbm_tc.ipynb`) và `quy_tac` (luật: `T − SpO₂ thấp nhất 10 phút cuối`).
- T = 89 và 91 không có trong train: AUPRC ở hai điểm này nằm trên đường nối các T lân cận thì model **nội suy được** sang ngưỡng chưa thấy.

In [ ]:
def rule_matrix(k):
    # luật: SpO2 thấp nhất 10 phút cuối càng thấp so với T thì rủi ro càng cao
    mn10 = np.nan_to_num(ST[k]["lv"]["mn_min10"].to_numpy(np.float32), nan=100.0)
    return (T_LIST[None, :] - mn10[:, None]).astype(np.float32)

def all_scores(k):
    S = {f"hybrid_s{s}": sig(seed_scores("main", s, k, NM)) for s in CFG["SEEDS"]}
    S["hybrid_ens"] = np.mean([S[f"hybrid_s{s}"] for s in CFG["SEEDS"]], 0)
    for n, p in CFG["BASELINES"].items():
        if os.path.exists(f"{p}_{k}.npz"):
            z = np.load(f"{p}_{k}.npz"); assert (z["T_list"] == T_LIST).all() and len(z["S"]) == len(DS[k]["X"]); S[n] = z["S"]
        else: print(f"Chưa có {p}_{k}.npz → chạy 10_lightgbm_tc.ipynb trước để có đối chứng '{n}'")
    S["quy_tac"] = rule_matrix(k); return S

dv = DS["val"]; S_val = all_scores("val")
MODELS = [n for n in ("hybrid_ens", "lgbm", "quy_tac") if n in S_val]; SEEDN = [f"hybrid_s{s}" for s in CFG["SEEDS"]]
R_val = pd.concat([E.by_T(n, S_val[n], dv, T_LIST, holdout=T_HO) for n in MODELS + SEEDN], ignore_index=True)
print("AUPRC trên val theo T:"); display(R_val.pivot(index="T", columns="model", values="AUPRC").round(4))
print("Trung bình AUPRC ở T train / T không train:")
display(R_val.groupby(["model", "holdout"])["AUPRC"].mean().unstack().round(4).rename(columns={False: "T train", True: "T không train"}))
E.plot_auprc_by_T(R_val[R_val.model.isin(MODELS)], T_HO, "VAL | G1")
E.plot_pr_roc_by_T({n: S_val[n] for n in MODELS}, dv, T_LIST, "VAL | G2")

### 5.2 Tính đơn điệu (G4), hiệu chuẩn (G5) và bảng ngưỡng theo cửa sổ
- **G4:** với mọi cửa sổ val, tính P ở mọi T rồi đếm tỷ lệ cặp vi phạm (T cao hơn mà P thấp hơn). LightGBM có ràng buộc đơn điệu nên phải bằng 0; hybrid chỉ được phạt mềm nên kỳ vọng gần 0.
- **G5:** đường hiệu chuẩn theo T. Xác suất do ta lấy mẫu con phía âm và có trọng số bù nên gần đúng về thang, nhưng chỉ cần cho hiển thị; quyết định cảnh báo dùng ngưỡng chọn theo báo giả.
- Bảng ngưỡng **đếm theo cửa sổ** (0,04 / 0,2 / 0,5 / 1 báo giả mỗi giờ) chỉ để tham khảo; ngưỡng thật chọn ở mục 7 bằng mô phỏng.

In [ ]:
MONO_V = pd.DataFrame({n: E.monotonic_report(S_val[n], T_LIST) for n in S_val if n != "quy_tac"}).T
print("Vi phạm đơn điệu trên val (adjacent: cặp T kề nhau; windows_any: cửa sổ có ít nhất một vi phạm):"); display(MONO_V.round(5))
E.plot_monotonic(S_val["hybrid_ens"], T_LIST, dv["pid"], title="VAL | G4 | hybrid_ens")
E.plot_reliability_by_T(S_val["hybrid_ens"], dv, T_LIST, title="VAL | G5 | hybrid_ens")
print("Ngưỡng xác suất theo T, chọn theo báo giả/giờ ĐẾM THEO CỬA SỔ (cận trên, chỉ tham khảo):")
display(E.thr_table(S_val["hybrid_ens"], dv, T_LIST, fa_levels=(0.04, 0.2, 0.5, 1)).round(4))

## 6. Ablation (G8)
Mỗi biến thể train **một seed** (`ABL_SEED`), so với model chính cùng seed:

| Biến thể | Khác model chính ở | Câu hỏi trả lời |
|---|---|---|
| `no_base` | bỏ 3 đặc trưng baseline | baseline của từng bệnh nhân có giúp không? |
| `abs` | đầu vào tuyệt đối (không trừ T; `frac_below` ở 90/92/95 cố định), T vẫn là đầu vào | biểu diễn tương đối có tốt hơn không? |
| `per_T` | 7 model riêng, mỗi model một T (như bản v2) | model chung có thua model riêng không? |

Model riêng không có ở T = 89, 91, nên đó chính là điểm yếu của cách cũ: muốn T mới thì phải train lại. Đặt `RUN_ABLATION=False` để bỏ qua (tốn thời gian nhất notebook).

In [ ]:
if CFG["RUN_ABLATION"]:
    s0 = CFG["ABL_SEED"]; ABL = {"main": sig(seed_scores("main", s0, "val", NM))}
    for nme, var in {"no_base": dict(rel=True, use_base=False), "abs": dict(rel=False, use_base=True)}.items():
        train_run(nme, s0, var); ABL[nme] = sig(seed_scores(nme, s0, "val", get_norm(var["rel"], var["use_base"])))
    S_pt = np.full_like(ABL["main"], np.nan)
    for T in T_TR:
        train_run(f"perT_{T}", s0, dict(rel=True, use_base=True, fixed_T=T))
        S_pt[:, C.col(dv, T)] = sig(seed_scores(f"perT_{T}", s0, "val", NM, [T]))[:, C.col(dv, T)]
    ABL["per_T"] = S_pt
    R_abl = pd.concat([E.by_T(n, S, dv, T_LIST, holdout=T_HO) for n, S in ABL.items()], ignore_index=True)
    display(R_abl.pivot(index="T", columns="model", values="AUPRC").round(4))
    E.plot_auprc_by_T(R_abl, T_HO, f"VAL | G8 ablation (seed {s0})")
    rows = []
    for n in ABL:
        if n == "main": continue
        for T in T_TR:
            j = C.col(dv, T); ok = dv["OK"][:, j]; lo, hi = E.paired_diff(ABL["main"][ok, j], ABL[n][ok, j], dv["Y"][ok, j], dv["pid"][ok])
            rows.append(dict(bien_the=n, T=T, chenh_lo=lo, chenh_hi=hi, ket_luan="main tốt hơn" if lo > 0 else "main kém hơn" if hi < 0 else "chưa phân biệt"))
    print("Chênh AUPRC (main − biến thể), CI 95% theo bệnh nhân:"); display(pd.DataFrame(rows).round(4))

## 7. Mô phỏng cảnh báo trên val

### 7.1 Timeline và chọn persistence
Mỗi T có một timeline riêng: sự kiện định nghĩa tại T (đã loại sự kiện nhiễu bằng HR, giống nhãn), điểm rủi ro là các cửa sổ `OK` tại T. `sweep_fa_day` so các cấu hình `k/n` và cooldown **ở cùng mức báo giả/ngày thật** (ngưỡng tự chọn để đạt ngân sách). Chọn cấu hình bắt được nhiều sự kiện nhất ở ngân sách mong muốn rồi ghi vào `CFG["TIERS"]`.

Lưu ý: timeline không có các cửa sổ ngay sau một sự kiện (đầu vào đã chứa sự kiện, phần đó do tầng Critical đảm nhiệm), nên báo giả/ngày có thể hơi lạc quan.

In [ ]:
CACHE = {"val": SA.load_records(dv["rec"], CFG["CLEAN_DIR"])}
def timelines(S, k, T_list=T_LIST):
    d = DS[k]
    return {int(T): SA.build_timelines(S[:, C.col(d, T)], d, T=int(T), cache=CACHE[k], clean_events=True,
                                       mask=d["OK"][:, C.col(d, T)]) for T in T_list}
TLv = {n: timelines(S_val[n], "val") for n in MODELS}
SW = pd.concat([SA.sweep_fa_day(TLv["hybrid_ens"][T], fa_days=(0.5, 1, 2, 3)).assign(T=T) for T in (88, 92) if T in TLv["hybrid_ens"]])
display(SW[["T"] + SA.SHOW].round(3))
for T, g in SW.groupby("T"): SA.plot_tradeoff(g, title=f"VAL | persistence và cooldown ở cùng mức báo giả | T = {T}")

### 7.2 Bảng T → ngưỡng (chọn trên val)
Với từng T **train** và từng tầng, chọn ngưỡng nhỏ nhất mà báo giả/ngày thật trên val không vượt ngân sách (`fa_day`). Sau đó làm mượt trên thang logit theo T (`THR_SMOOTH`) và nội suy cho mọi T nguyên trong `T_RANGE`. Ngưỡng ở T = 89, 91 vì thế đến từ **nội suy**, đúng như khi bác sĩ đặt một T mới. Mỗi đối chứng (LightGBM, luật) có bảng riêng ở cùng ngân sách, để so sánh công bằng.

In [ ]:
T_OUT = range(CFG["T_RANGE"][0], CFG["T_RANGE"][1] + 1)
def stream_thr_table(TL, T_list=T_TR):
    return pd.DataFrame({t: {int(T): SA.thr_for_fa_day(TL[int(T)], c["fa_day"], c["k"], c["n"], c["cooldown"]) for T in T_list}
                         for t, c in CFG["TIERS"].items()}).rename_axis("T")
THR_RAW = {n: stream_thr_table(TLv[n]) for n in MODELS}
THR = {n: (E.smooth_table(THR_RAW[n], T_OUT, CFG["THR_SMOOTH"]) if n != "quy_tac" else
           THR_RAW[n].reindex(list(T_OUT)).interpolate(limit_direction="both")) for n in MODELS}    # luật: điểm không phải xác suất
display(pd.concat({"thô (val)": THR_RAW["hybrid_ens"], "làm mượt": THR["hybrid_ens"]}, axis=1).round(4))
fig, ax = plt.subplots(figsize=(7, 4))
for i, t in enumerate(CFG["TIERS"]):
    ax.plot(THR_RAW["hybrid_ens"].index, E.logit(THR_RAW["hybrid_ens"][t]), "o", c=f"C{i}", label=f"{t} (thô)")
    ax.plot(THR["hybrid_ens"].index, E.logit(THR["hybrid_ens"][t]), "-", c=f"C{i}", label=f"{t} (làm mượt)")
for t in T_HO: ax.axvline(t, c="gray", ls=":")
ax.set(xlabel="T (%)", ylabel="ngưỡng (logit)", title="Bảng T → ngưỡng của hybrid_ens"); ax.legend(); plt.tight_layout(); plt.show()

def tier_table(TL, thr_tab, T_list=T_LIST):
    rows, leads, pts = [], {}, {}
    for T in T_list:
        for t, c in CFG["TIERS"].items():
            s, pt, l = SA.run_stream(TL[int(T)], float(thr_tab.loc[int(T), t]), c["k"], c["n"], c["cooldown"])
            rows.append(dict(T=int(T), tier=t, holdout=int(T) in T_HO, **s))
            if t == "alert": leads[int(T)], pts[int(T)] = l, pt
    return pd.DataFrame(rows), leads, pts
G6_COLS = ["T", "tier", "holdout", "alerts_per_day", "FA_per_day", "FA_day_pt_p90", "precision", "caught_useful", "lead_med", "evaluable"]
TV, LV, _ = tier_table(TLv["hybrid_ens"], THR["hybrid_ens"])
print("VAL | hybrid_ens với bảng ngưỡng đã làm mượt (kiểm tra ngân sách; T không train dùng ngưỡng nội suy):"); display(TV[G6_COLS].round(3))

### 7.3 G3 trên val: AI hơn luật ở điểm nào?
Với mỗi model, mỗi T và mỗi ngân sách trong `FA_GRID`: chọn ngưỡng trên val (cấu hình tầng Alert) rồi đo tỷ lệ sự kiện bắt được với báo trước ≥ 5 phút. Đường của model nằm **trên** đường của luật nghĩa là ở **cùng số báo giả/ngày**, model bắt được nhiều sự kiện hơn.

In [ ]:
CA = CFG["TIERS"]["alert"]
G3_THR = {(n, T, fa): SA.thr_for_fa_day(TLv[n][T], fa, CA["k"], CA["n"], CA["cooldown"])
          for n in MODELS for T in TLv[n] for fa in CFG["FA_GRID"]}
def g3_table(TL):
    return pd.DataFrame([dict(model=n, T=T, fa_budget=fa, **SA.run_stream(TL[n][T], th, CA["k"], CA["n"], CA["cooldown"])[0])
                         for (n, T, fa), th in G3_THR.items()])
def plot_g3(df, title):
    Ts = sorted(df["T"].unique()); fig, ax = plt.subplots(1, len(Ts), figsize=(3.1 * len(Ts), 3.6), sharey=True, squeeze=False)
    for a, T in zip(ax[0], Ts):
        for n, g in df[df["T"] == T].groupby("model", sort=False): a.plot(g["FA_per_day"], g["caught_useful"] * 100, "o-", ms=3, label=n)
        a.set(xscale="log", xlabel="Báo giả/ngày (thật)", title=f"T = {T}" + (" (không train)" if T in T_HO else ""))
    ax[0][0].set_ylabel("% sự kiện bắt được, báo trước ≥ 5 phút"); ax[0][0].legend(fontsize=7)
    fig.suptitle(title); plt.tight_layout(); plt.show()
G3V = g3_table(TLv); plot_g3(G3V, "VAL | G3 | tầng Alert")
display(G3V.pivot_table(index=["T", "fa_budget"], columns="model", values="caught_useful").round(3))

## 8. Chốt cấu hình
Trước khi mở test, ghi lại mọi lựa chọn đã làm trên val: seed triển khai, cấu hình tầng, phương pháp làm mượt bảng ngưỡng. Từ đây **không đổi gì** sau khi đã xem test.

In [ ]:
DEPLOY = max(CFG["SEEDS"], key=lambda s: RUNS[s]["best"])
FROZEN = dict(deploy_seed=int(DEPLOY), tiers=CFG["TIERS"], thr_smooth=CFG["THR_SMOOTH"], lambda_mono=CFG["LAMBDA_MONO"],
              thr_table_hybrid_ens=THR["hybrid_ens"].round(6).to_dict(), time=time.strftime("%Y-%m-%d %H:%M"))
json.dump(FROZEN, open(f"{ROOT}/frozen_config.json", "w", encoding="utf-8"), indent=1, default=str)
print(json.dumps({k: v for k, v in FROZEN.items() if k != "thr_table_hybrid_ens"}, indent=1, default=str))

## 9. Đánh giá cuối trên test (chạy một lần)
Chỉ chạy sau mục 8. Ngưỡng, persistence, cooldown, seed đều giữ nguyên từ val.

### 9.1 Phân biệt: G1, G2 và so sánh cặp theo T
Chênh AUPRC (hybrid_ens − đối chứng) lấy mẫu lại theo bệnh nhân: khoảng không chứa 0 thì khác biệt có ý nghĩa.

In [ ]:
# CHẠY MỘT LẦN
dt = DS["test"]; S_test = all_scores("test")
R_test = pd.concat([E.by_T(n, S_test[n], dt, T_LIST, holdout=T_HO) for n in MODELS + SEEDN], ignore_index=True)
print("AUPRC trên test theo T:"); display(R_test.pivot(index="T", columns="model", values="AUPRC").round(4))
display(R_test[R_test.model.isin(MODELS)][["model", "T", "holdout", "AUPRC", "lo", "hi", "lift", "AUROC", "prevalence", "n_pos", "pat_pos"]].round(4))
E.plot_auprc_by_T(R_test[R_test.model.isin(MODELS)], T_HO, "TEST | G1")
E.plot_pr_roc_by_T({n: S_test[n] for n in MODELS}, dt, T_LIST, "TEST | G2")
PD = []
for n in MODELS[1:]:
    for T in T_LIST:
        j = C.col(dt, T); ok = dt["OK"][:, j]; lo, hi = E.paired_diff(S_test["hybrid_ens"][ok, j], S_test[n][ok, j], dt["Y"][ok, j], dt["pid"][ok])
        PD.append(dict(doi_chung=n, T=int(T), lo=lo, hi=hi, ket_luan="hybrid tốt hơn" if lo > 0 else "hybrid kém hơn" if hi < 0 else "chưa phân biệt"))
PD = pd.DataFrame(PD); print("Chênh AUPRC (hybrid_ens − đối chứng), CI 95% theo bệnh nhân:"); display(PD.round(4))

### 9.2 Đơn điệu (G4) và hiệu chuẩn (G5)
Isotonic theo T được fit trên **val** và áp lên test (bảng phải). Isotonic chỉ để hiển thị trên dashboard.

In [ ]:
display(pd.DataFrame({n: E.monotonic_report(S_test[n], T_LIST) for n in S_test if n != "quy_tac"}).T.round(5))
E.plot_monotonic(S_test["hybrid_ens"], T_LIST, dt["pid"], title="TEST | G4 | hybrid_ens")
CAL = E.fit_calibrators(S_val["hybrid_ens"], dv, T_LIST)
E.plot_reliability_by_T(S_test["hybrid_ens"], dt, T_LIST, cal=CAL, title="TEST | G5 | hybrid_ens")

### 9.3 Vận hành: G6, G7, G3 trên test
- **G6:** mỗi T, mỗi tầng: cảnh báo/ngày, báo giả/ngày (trung bình và phân vị 90 theo bệnh nhân), precision, tỷ lệ bắt được với báo trước ≥ 5 phút, báo trước trung vị. Mỗi model dùng bảng ngưỡng **của chính nó** chọn trên val ở cùng ngân sách.
- **G7:** histogram thời gian báo trước theo T và phân phối báo giả theo bệnh nhân (tầng Alert).
- **G3:** như mục 7.3, ngưỡng từ val, đo trên test.

In [ ]:
CACHE["test"] = SA.load_records(dt["rec"], CFG["CLEAN_DIR"])
TLt = {n: timelines(S_test[n], "test") for n in MODELS}
G6, LEADS, PTS = {}, {}, {}
for n in MODELS: G6[n], LEADS[n], PTS[n] = tier_table(TLt[n], THR[n])
G6T = pd.concat({n: g[G6_COLS] for n, g in G6.items()}, names=["model", None]).reset_index(level=0)
print("TEST | G6:"); display(G6T.round(3))
SA.plot_by_T(G6["hybrid_ens"], LEADS["hybrid_ens"], "TEST | G6/G7 | hybrid_ens")
for T in (88, 92):
    if T in PTS["hybrid_ens"]: SA.plot_config(PTS["hybrid_ens"][T], LEADS["hybrid_ens"][T], f"TEST | G7 | hybrid_ens | tầng Alert | T = {T}")
G3T = g3_table(TLt); plot_g3(G3T, "TEST | G3 | tầng Alert (ngưỡng chọn trên val)")
display(G3T.pivot_table(index=["T", "fa_budget"], columns="model", values=["FA_per_day", "caught_useful"]).round(3))
w_, a_ = CFG["TIERS"]["watch"], CFG["TIERS"]["alert"]; Tx = 92 if 92 in TLt["hybrid_ens"] else int(T_LIST[-1])
t0_ = TLt["hybrid_ens"][Tx]; rec0 = max(t0_, key=lambda r: len(t0_[r]["onsets"]) if len(t0_[r]["t"]) > 50 else -1)
SA.plot_example(t0_, THR["hybrid_ens"].loc[Tx, "watch"], THR["hybrid_ens"].loc[Tx, "alert"],
                (w_["k"], w_["n"], w_["cooldown"]), (a_["k"], a_["n"], a_["cooldown"]), rec=rec0)

### 9.4 Tầng Critical (G10)
Critical là **luật**, không dùng ML: SpO₂ < ngưỡng Critical liên tục `n_min` phút với SQI (`spo2_valid`) tốt, hoặc mạch ngoài `[hr_low, hr_high]` liên tục `n_min` phút. Bảng cho thấy gánh nặng cảnh báo của luật theo từng ngưỡng Critical, có và không có luật nhịp tim. Thống kê trên toàn bộ phút có dữ liệu của các record test.

In [ ]:
tl_all = {r: dict(pid=pid_, spo2=c["spo2_mean"], valid=c["spo2_valid"], pulse=c["pulse_mean"], onsets=np.array([], int),
                  valid_h=np.isfinite(c["spo2_mean"]).sum() / 60)
          for r, c in CACHE["test"].items() for pid_ in [dt["pid"][dt["rec"] == r][0]]}
cr = CFG["CRIT"]; G10 = []
for ct in cr["crit_T"]:
    for use_hr in (False, True):
        s = SA.critical_stats(tl_all, cr["cooldown"], crit_T=ct, n_min=cr["n_min"], sqi_min=cr["sqi_min"],
                              hr_low=cr["hr_low"] if use_hr else None, hr_high=cr["hr_high"] if use_hr else None)
        G10.append(dict(crit_T=ct, luat_HR=use_hr, **s))
G10 = pd.DataFrame(G10); print("TEST | G10 | tầng Critical (luật):"); display(G10.round(3))
fig, ax = plt.subplots(figsize=(7, 4))
for use_hr, g in G10.groupby("luat_HR"):
    ax.plot(g["crit_T"], g["alerts_per_day"], "o-", label=("có" if use_hr else "không có") + " luật nhịp tim (TB)")
    ax.plot(g["crit_T"], g["FA_day_pt_p90"], "x--", label=("có" if use_hr else "không có") + " luật nhịp tim (p90 BN)")
ax.set(xlabel="Ngưỡng Critical (%)", ylabel="Cảnh báo Critical / ngày", title="G10: gánh nặng của tầng Critical"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

## 10. Xuất model cho ESP32-C3 (models_v3)

| File | Nội dung |
|---|---|
| `hybrid_tc.keras` | model Keras (seed triển khai) |
| `hybrid_tc_int8.tflite` | INT8, hai đầu vào (`seq`, `tab`), đầu ra logit |
| `hybrid_tc_norm.npz` | `seq_mu`, `seq_sd` (thang tương đối), `tab_cols`, `tab_med`, `tab_mu`, `tab_sd`, `fb_grid`, `base_min_hours` |
| `hybrid_tc_meta.json` | thứ tự đầu vào, thang lượng tử hóa, **bảng T → ngưỡng** (xác suất, logit, int8) cho Watch/Alert, khoảng T hợp lệ, persistence, cooldown, cấu hình Critical và baseline, `model_version` |

**Quyết định kỹ thuật**
- Dữ liệu đại diện cho lượng tử hóa: 1.200 mẫu ngẫu nhiên + 300 mẫu dương, mỗi mẫu một T ngẫu nhiên, truyền dạng **dict theo tên đầu vào** (converter sắp xếp đầu vào theo tên; firmware đưa dữ liệu đúng `input_order`).
- **Bảng ngưỡng cho thiết bị chọn lại trên điểm INT8 của val** (đúng thứ thiết bị tính), cùng quy trình mục 7.2.
- Firmware: chuỗi `(x − [T, T, 0, 0] − seq_mu)/seq_sd`, NaN → 0, thêm 2 mask; bảng đặc trưng tính như `tc_core.make_features`, điền `tab_med`, z-score, cắt ±5; lượng tử hóa theo `scale`, `zero_point`. Server gửi bảng ngưỡng trong config `dev/{id}/config` có version; web chặn T ngoài `T_valid_range`.
- G9: so logit float và INT8 trên val (scatter) và chênh AUPRC theo T.

In [ ]:
EXP, NAME = CFG["EXPORT_DIR"], CFG["NAME"]; os.makedirs(EXP, exist_ok=True)
best = load_best("main", DEPLOY); best.save(f"{EXP}/{NAME}.keras")
np.savez(f"{EXP}/{NAME}_norm.npz", **{k: np.asarray(v) for k, v in NM.items()}, fb_grid=C.FB_GRID, base_min_hours=C.BASE_MIN_HOURS)

rng = np.random.default_rng(0); dtr = DS["train"]
ri, rT, ry, _ = C.sample_epoch(dtr, T_TR, 1.0, rng)
rep_i, rep_T = np.r_[ri[:1200], ri[ry == 1][:300]], np.r_[rT[:1200], rT[ry == 1][:300]]
REP = C.inputs(dtr, ST["train"], NM, rep_T, rep_i)
conv = tf.lite.TFLiteConverter.from_keras_model(best)
conv.optimizations = [tf.lite.Optimize.DEFAULT]
conv.representative_dataset = lambda: ({"seq": REP["seq"][i:i + 1], "tab": REP["tab"][i:i + 1]} for i in range(len(rep_i)))
conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
conv.inference_input_type = conv.inference_output_type = tf.int8
TFL = f"{EXP}/{NAME}_int8.tflite"; open(TFL, "wb").write(conv.convert())

def interp(batch):
    it = tf.lite.Interpreter(model_path=TFL)
    for x in it.get_input_details(): it.resize_tensor_input(x["index"], [batch] + list(x["shape"][1:]))
    it.allocate_tensors(); return it
IT1 = interp(1); det = IT1.get_input_details(); d_seq = next(x for x in det if len(x["shape"]) == 3); d_tab = next(x for x in det if len(x["shape"]) == 2)
osc, ozp = IT1.get_output_details()[0]["quantization"]
q = lambda x, dd: np.clip(np.round(x / dd["quantization"][0] + dd["quantization"][1]), -128, 127).astype(np.int8)
def tflite_logit(X, it=IT1, B=1):
    di = {len(x["shape"]): x["index"] for x in it.get_input_details()}; oi = it.get_output_details()[0]["index"]
    n = len(X["seq"]); out = np.empty(n, np.float32)
    for a in range(0, n, B):
        s_, t_ = X["seq"][a:a + B], X["tab"][a:a + B]; m = len(s_)
        if m < B: s_, t_ = np.concatenate([s_, np.repeat(s_[-1:], B - m, 0)]), np.concatenate([t_, np.repeat(t_[-1:], B - m, 0)])
        it.set_tensor(di[3], q(s_, d_seq)); it.set_tensor(di[2], q(t_, d_tab)); it.invoke()
        out[a:a + m] = (it.get_tensor(oi)[:m, 0].astype(np.float32) - ozp) * osc
    return out
try:                                                     # chạy theo lô cho nhanh; kiểm tra khớp với lô 1
    ITB = interp(256); chk = {k: v[:300] for k, v in REP.items()}
    assert np.allclose(tflite_logit(chk, ITB, 256), tflite_logit(chk)); run_q = lambda X: tflite_logit(X, ITB, 256)
except Exception as e:
    print("Không chạy theo lô được, dùng lô 1:", e); run_q = tflite_logit

def int8_matrix(k):
    Z = np.full((len(DS[k]["X"]), len(T_LIST)), np.nan, np.float32)
    for T in tqdm(T_LIST, desc=f"INT8 {k}", leave=False): Z[:, C.col(DS[k], T)] = run_q(C.inputs(DS[k], ST[k], NM, int(T)))
    return Z
Zq_val = int8_matrix("val"); Zf_val = seed_scores("main", DEPLOY, "val", NM)
kb = os.path.getsize(TFL) / 1024
print(f"Seed triển khai: {DEPLOY} | TFLite INT8: {kb:.1f} KB (mục tiêu < 30 KB) | tham số: {best.count_params():,} | bước logit đầu ra: {osc:.4f}")

In [ ]:
# G9: INT8 so với float trên val
G9 = pd.concat([E.by_T("float", sig(Zf_val), dv, T_LIST, n_boot=0), E.by_T("int8", sig(Zq_val), dv, T_LIST, n_boot=0)])
G9 = G9.pivot(index="T", columns="model", values="AUPRC"); G9["chenh_tuong_doi"] = (G9["int8"] - G9["float"]) / G9["float"]
display(G9.round(4))
fig, ax = plt.subplots(1, 2, figsize=(13, 4.4)); rs_ = np.random.default_rng(0)
ii = rs_.choice(len(Zf_val), min(20000, len(Zf_val)), replace=False); jj = rs_.integers(0, len(T_LIST), len(ii))
ax[0].scatter(Zf_val[ii, jj], Zq_val[ii, jj], s=2, alpha=.3); lim = [np.nanmin(Zf_val), np.nanmax(Zf_val)]; ax[0].plot(lim, lim, "r--", lw=.8)
ax[0].set(xlabel="logit float", ylabel="logit INT8", title=f"G9: logit float vs INT8 (val, T ngẫu nhiên) | r = {np.corrcoef(Zf_val[ii, jj], Zq_val[ii, jj])[0, 1]:.4f}")
ax[1].bar(G9.index.astype(str), G9["chenh_tuong_doi"] * 100); ax[1].axhline(-5, c="r", ls="--", lw=.8, label="mức chấp nhận −5%")
ax[1].set(xlabel="T", ylabel="% chênh AUPRC (INT8 − float)", title="G9: suy giảm do lượng tử hóa theo T"); ax[1].legend()
plt.tight_layout(); plt.show()

# bảng ngưỡng cho thiết bị: chọn lại trên điểm INT8 của val (cùng quy trình mục 7.2)
if "val" not in CACHE: CACHE["val"] = SA.load_records(dv["rec"], CFG["CLEAN_DIR"])
THR_DEV = E.smooth_table(stream_thr_table(timelines(sig(Zq_val), "val")), T_OUT, CFG["THR_SMOOTH"])
lg = THR_DEV.apply(E.logit); i8 = np.clip(np.floor(lg / osc + ozp), -128, 127).astype(int)    # cảnh báo khi int8_out > ngưỡng_int8
meta = dict(
    model_version=f"{NAME}-{time.strftime('%Y%m%d')}-s{DEPLOY}", deploy_seed=int(DEPLOY), tflite_kb=round(kb, 1), params=int(best.count_params()),
    input_order=[x["name"] for x in det],
    input_seq=dict(shape=[C.IN_MIN, 6], channels=["spo2_mean-T", "spo2_min-T", "spo2_valid", "pulse_mean", "mask_spo2_mean", "mask_pulse_mean"],
                   scale=float(d_seq["quantization"][0]), zero_point=int(d_seq["quantization"][1])),
    input_tab=dict(shape=[len(NM["tab_cols"])], features=[str(c) for c in NM["tab_cols"]],
                   scale=float(d_tab["quantization"][0]), zero_point=int(d_tab["quantization"][1])),
    output=dict(scale=float(osc), zero_point=int(ozp), kind="logit"),
    T_valid_range=list(CFG["T_RANGE"]), T_train=T_TR, T_holdout=T_HO,
    thresholds=dict(T=[int(t) for t in THR_DEV.index],
                    **{f"{t}_prob": THR_DEV[t].round(6).tolist() for t in THR_DEV}, **{f"{t}_logit": lg[t].round(4).tolist() for t in lg},
                    **{f"{t}_int8": i8[t].tolist() for t in i8}, selected_on="val INT8, báo giả/ngày thật sau persistence+cooldown",
                    smoothing=CFG["THR_SMOOTH"]),
    tiers=CFG["TIERS"], critical=CFG["CRIT_DEPLOY"],
    baseline=dict(definition="trung vị spo2_mean các phút SQI tốt trong 24-72 giờ gần nhất (firmware); train dùng 6-24 giờ trước cửa sổ",
                  min_hours=C.BASE_MIN_HOURS, fallback="SpO2 trung bình của cửa sổ hiện tại, base_hours=0",
                  freeze_during_alert=True, clinician_reference=True),
    unknown_state=dict(rule="spo2_valid trung bình 10 phút cuối < 0.5 hoặc thiếu > 30% phút đầu vào -> UNKNOWN, không chạy model"),
)
json.dump(meta, open(f"{EXP}/{NAME}_meta.json", "w", encoding="utf-8"), indent=1, ensure_ascii=False)
print("Đã xuất:", sorted(f for f in os.listdir(EXP) if f.startswith(NAME))); display(pd.concat({"prob": THR_DEV, "int8": i8}, axis=1).round(4))

## 11. Suy luận thử bằng bản INT8
Mô phỏng đúng việc firmware làm với một cửa sổ test: tính đặc trưng ở mọi T nguyên trong `T_RANGE` (kể cả 85 và 95 chưa từng train), chạy bản `.tflite`, so với bảng ngưỡng trong `meta.json`. Đường P theo T phải không giảm; nếu bác sĩ đổi T từ 88 lên 92 thì cùng một cửa sổ có thể chuyển từ "bình thường" sang WATCH/ALERT mà không cần train lại.

In [ ]:
Zq_test = int8_matrix("test"); TT_ = list(T_OUT)
print(f"AUPRC test trung bình (T train) | float seed {DEPLOY}: {E.mean_auprc(seed_scores('main', DEPLOY, 'test', NM), dt, T_TR):.4f} "
      f"| INT8: {E.mean_auprc(Zq_test, dt, T_TR):.4f}")
cand = np.where(dt["Y"][:, C.col(dt, 92)] & dt["OK"][:, C.col(dt, 92)] & ~dt["Y"][:, C.col(dt, 88)].astype(bool))[0]
pick = list(np.random.default_rng(3).choice(cand, min(3, len(cand)), replace=False)) if len(cand) else [0]
fig, ax = plt.subplots(1, len(pick), figsize=(6 * len(pick), 4), squeeze=False); rows = []
for a, i in zip(ax[0], pick):
    P = sig(np.array([run_q(C.inputs(dt, ST["test"], NM, T, [i]))[0] for T in TT_]))
    tier = ["ALERT" if p > THR_DEV.loc[T, "alert"] else "WATCH" if p > THR_DEV.loc[T, "watch"] else "—" for p, T in zip(P, TT_)]
    a.plot(TT_, P, "o-", label="P (INT8)"); a.plot(TT_, THR_DEV["watch"], ":", c="orange", label="ngưỡng Watch"); a.plot(TT_, THR_DEV["alert"], ":", c="r", label="ngưỡng Alert")
    a.set(yscale="log", xlabel="T do bác sĩ đặt (%)", ylabel="P", title=f"{dt['rec'][i]} @ {dt['start'][i]} | đơn điệu: {bool(np.all(np.diff(P) >= -1e-6))}"); a.legend(fontsize=8)
    rows.append({"rec": dt["rec"][i], **{f"T={T}": t for T, t in zip(TT_, tier)}})
plt.tight_layout(); plt.show(); display(pd.DataFrame(rows))

## 12. Tổng kết

### 12.1 Tổng kết tự động

In [ ]:
rv = R_test[R_test.model == "hybrid_ens"].set_index("T"); tr_m, ho_m = rv.loc[T_TR, "AUPRC"].mean(), rv.loc[T_HO, "AUPRC"].mean()
nb_ = {t: rv.loc[[t - 1, t + 1], "AUPRC"].mean() for t in T_HO if t - 1 in rv.index and t + 1 in rv.index}
print("TỔNG KẾT (số liệu tự sinh từ các cell phía trên)\n")
print(f"1. Một model cho mọi T: AUPRC test trung bình {tr_m:.3f} ở T train, {ho_m:.3f} ở T không train "
      + "(" + ", ".join(f"T={t}: {rv.loc[t, 'AUPRC']:.3f} so với TB hai T kề {v:.3f}" for t, v in nb_.items()) + ").")
print(f"   Hệ số so với ngẫu nhiên: {rv['lift'].min():.1f}–{rv['lift'].max():.1f} lần.")
for n in MODELS[1:]:
    g = PD[PD.doi_chung == n]; print(f"2. So với {n}: hybrid tốt hơn ở {(g.ket_luan == 'hybrid tốt hơn').sum()}/{len(g)} T, "
                                      f"kém hơn ở {(g.ket_luan == 'hybrid kém hơn').sum()}, chưa phân biệt ở {(g.ket_luan == 'chưa phân biệt').sum()}.")
mv = E.monotonic_report(S_test["hybrid_ens"], T_LIST); print(f"3. Đơn điệu theo T: {mv['adjacent']:.2%} cặp kề nhau vi phạm, {mv['windows_any']:.2%} cửa sổ có vi phạm.")
for T in (88, 92):
    if T in LEADS["hybrid_ens"]:
        h = G6["hybrid_ens"].query("T == @T and tier == 'alert'").iloc[0]
        r_ = G6["quy_tac"].query("T == @T and tier == 'alert'").iloc[0] if "quy_tac" in G6 else None
        print(f"4. Tầng Alert @ T={T}: bắt {h.caught_useful:.0%} sự kiện (báo trước ≥ 5 phút) với {h.FA_per_day:.2f} báo giả/ngày, precision {h.precision:.0%}, "
              f"báo trước trung vị {h.lead_med:.0f} phút" + (f"; luật cùng ngân sách: {r_.caught_useful:.0%} với {r_.FA_per_day:.2f}/ngày." if r_ is not None else "."))
print(f"5. INT8: {meta['tflite_kb']} KB, chênh AUPRC val lớn nhất theo T {G9['chenh_tuong_doi'].min():.1%}" + (" → cần xem lại lượng tử hóa." if G9['chenh_tuong_doi'].min() < -0.05 else " (trong mức 5%)."))

### 12.2 Lưu ý trung thực khi báo cáo
- **Domain shift:** MIMIC là bệnh nhân ICU đo bằng đầu kẹp ngón tay; vòng đeo đo phản xạ ở cổ tay (MAX30102). MIMIC chỉ dùng để tiền huấn luyện và chứng minh khái niệm.
- **AUPRC tuyệt đối thấp** vì lớp dương hiếm, nhất là ở T thấp; luôn báo cáo kèm hệ số so với ngẫu nhiên và so với luật ở **cùng mức báo giả/ngày**.
- Nếu hybrid vẫn chưa phân biệt được với LightGBM, giá trị cần nhấn mạnh là: một model cho nhiều ngưỡng (kể cả ngưỡng chưa thấy), đơn điệu theo T, đủ nhỏ để chạy trên thiết bị, và hơn luật ở cùng mức báo giả.
- Tầng dự đoán chỉ phủ sự kiện sau một đoạn đầu vào sạch; sự kiện nối tiếp do tầng Critical xử lý. Báo giả/ngày có thể hơi lạc quan do timeline bỏ các cửa sổ ngay sau sự kiện.
- Chưa có mã ICD-9 nên chưa đánh giá theo nhóm bệnh (COPD và không COPD).
- Bộ nhớ (tensor arena) và thời gian suy luận trên ESP32-C3 cần đo trực tiếp trên thiết bị.